# Dependent SBM: Residual Mean vs Median Ratio, T = 200

Simulation prototype for comparing three KAP-CPD variants across rho values:

- `K_resid_raw_mean`
- `K_ratio_median`
- original unadjusted `K`

For each rho, the notebook runs 100 independent repeats. For each repeat and method it records:

- `detected`: permutation p-value <= 0.05
- `accurate_detected`: detected and `abs(tauhat - tau) <= 0.05 * T`

The final summary reports the mean of those indicators across repeats.

In [ ]:
import os
import sys
import time
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

LOCAL_PROJECT_ROOT = Path('/Users/sophiasun/networkcp/Dependent_networks/python')
COLAB_PROJECT_ROOT = Path('/content/dependent_network/python')

if LOCAL_PROJECT_ROOT.exists():
    PROJECT_ROOT = LOCAL_PROJECT_ROOT
    RUNTIME_ENV = 'local'
elif COLAB_PROJECT_ROOT.exists():
    PROJECT_ROOT = COLAB_PROJECT_ROOT
    RUNTIME_ENV = 'colab'
else:
    raise FileNotFoundError(
        'Could not find the project root. On Colab, run:\n'
        '%cd /content\n'
        '!git clone https://github.com/sophiasun025/dependent_network.git\n'
        '%cd /content/dependent_network/python'
    )

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import dependent_networks.dependent_network_functions as dnf
importlib.reload(dnf)

print('Runtime environment:', RUNTIME_ENV)
print('Project root:', PROJECT_ROOT)
print('Using module:', dnf.__file__)

In [ ]:
# Simulation settings
RHO_VALUES = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
SIGNAL = 0.05
T = 200
TAU = 70
BLOCK_NUMS = [10, 10, 10, 10, 10]
N_REPEATS = 100
ALPHA = 0.05
ACCURACY_TOL = 0.05 * T

# Permutation settings. Use B_PERM=1000 for final runs.
# BATCH_SIZE keeps JAX permutation memory bounded on Colab/GPU.
B_PERM = 1000
BATCH_SIZE = 50

# Lag adjustment setting
MAX_LAG = T

# Seeds are deterministic/reproducible and unique per rho/repeat.
BASE_SEED = 20260601

METHODS = [
    {'name': 'K_resid_raw_mean', 'adjust_type': 'K_resid_raw_mean'},
    {'name': 'K_ratio_median', 'adjust_type': 'K_ratio_median'},
    {'name': 'original_K', 'adjust_type': None},
]

# Local writes to the repo. Colab writes to Google Drive when mounted, so results
# survive runtime resets; otherwise it falls back to the cloned repo under /content.
if IN_COLAB and Path('/content/drive/MyDrive').exists():
    RESULTS_DIR = Path('/content/drive/MyDrive/dependent_network_results')
else:
    RESULTS_DIR = PROJECT_ROOT / 'notebooks' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DETAIL_PATH = RESULTS_DIR / f'dependent_SBM_resid_medianratio_T={T}_detail.csv'
SUMMARY_PATH = RESULTS_DIR / f'dependent_SBM_resid_medianratio_T={T}_summary.csv'

print('Detail path:', DETAIL_PATH)
print('Summary path:', SUMMARY_PATH)

In [ ]:
def simulation_seed(rho_index: int, repeat_index: int) -> int:
    return BASE_SEED + 10_000 * rho_index + repeat_index


def simulate_once(rho: float, seed: int):
    return dnf.generate_dependent_sbm_cp_jax(
        rho=rho,
        p_within_before=0.3,
        p_within_after=0.3 + SIGNAL,
        p_across_before=0.1,
        p_across_after=0.1,
        block_nums=BLOCK_NUMS,
        tau=TAU,
        T=T,
        seed=seed,
        include_graphlet_kernel=True,
    )


def evaluate_method(K1_raw, K2_raw, method: dict, seed: int) -> dict:
    method_name = method['name']
    adjust_type = method['adjust_type']

    if adjust_type is None:
        stat = dnf.kap_cpd_statistic(K1_raw, K2_raw, n=T)
        pvalue = dnf.kap_cpd_permutation_pvalue_batched(
            K1_raw,
            K2_raw,
            B=B_PERM,
            batch_size=BATCH_SIZE,
            n=T,
            seed=seed,
        )
    else:
        stat = dnf.kap_cpd_statistic_dependent(
            K1_raw,
            K2_raw,
            n=T,
            adjust_type=adjust_type,
            max_lag=MAX_LAG,
        )
        pvalue = dnf.kap_cpd_permutation_pvalue_dependent_batched(
            K1_raw,
            K2_raw,
            adjust_type=adjust_type,
            max_lag=MAX_LAG,
            B=B_PERM,
            batch_size=BATCH_SIZE,
            n=T,
            seed=seed,
        )

    tauhat = stat['S']['tauhat']
    detected = pvalue <= ALPHA
    accurate_detected = detected and (abs(tauhat - TAU) <= ACCURACY_TOL)

    return {
        'method': method_name,
        'pvalue': float(pvalue),
        'tauhat': int(tauhat),
        'S_max': float(stat['S']['max']),
        'detected': bool(detected),
        'accurate_detected': bool(accurate_detected),
    }


def load_existing_results() -> pd.DataFrame:
    if DETAIL_PATH.exists():
        return pd.read_csv(DETAIL_PATH)
    return pd.DataFrame()


def append_rows(rows: list[dict]) -> None:
    df = pd.DataFrame(rows)
    write_header = not DETAIL_PATH.exists()
    df.to_csv(DETAIL_PATH, mode='a', header=write_header, index=False)


def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        df.groupby(['rho', 'method'], as_index=False)
        .agg(
            repeats=('repeat', 'nunique'),
            detection=('detected', 'mean'),
            accurate_detection=('accurate_detected', 'mean'),
            mean_pvalue=('pvalue', 'mean'),
            median_pvalue=('pvalue', 'median'),
            mean_tauhat=('tauhat', 'mean'),
            median_tauhat=('tauhat', 'median'),
        )
        .sort_values(['rho', 'method'])
    )
    return summary

In [ ]:
# Run simulation. This cell is resume-friendly: completed rho/repeat/method rows are skipped.
existing = load_existing_results()
completed = set()
if not existing.empty:
    completed = set(zip(existing['rho'], existing['repeat'], existing['method']))
    print(f'Resuming from {len(existing)} existing method-level rows.')

start_time = time.time()

for rho_index, rho in enumerate(tqdm(RHO_VALUES, desc='rho')):
    for repeat_index in tqdm(range(N_REPEATS), desc=f'repeats rho={rho}', leave=False):
        pending_methods = [
            method for method in METHODS
            if (rho, repeat_index, method['name']) not in completed
        ]
        if not pending_methods:
            continue

        seed = simulation_seed(rho_index, repeat_index)
        toy_data = simulate_once(rho=rho, seed=seed)
        K1_raw = toy_data.K1
        K2_raw = toy_data.K2

        rows = []
        for method_index, method in enumerate(pending_methods):
            method_seed = seed + 1_000_000 * (method_index + 1)
            result = evaluate_method(K1_raw, K2_raw, method, seed=method_seed)
            rows.append({
                'rho': rho,
                'repeat': repeat_index,
                'seed': seed,
                'T': T,
                'tau': TAU,
                'signal': SIGNAL,
                'B_perm': B_PERM,
                'batch_size': BATCH_SIZE,
                'max_lag': MAX_LAG,
                **result,
            })

        append_rows(rows)
        completed.update((row['rho'], row['repeat'], row['method']) for row in rows)

elapsed = time.time() - start_time
print(f'Done or resumed all requested rows in {elapsed / 60:.2f} minutes.')

In [ ]:
# Aggregate detection metrics across repeats.
detail = pd.read_csv(DETAIL_PATH)
summary = summarize_results(detail)
summary.to_csv(SUMMARY_PATH, index=False)
summary

In [ ]:
# Quick pivot table for the two requested metrics.
summary.pivot(index='rho', columns='method', values=['detection', 'accurate_detection'])

In [ ]:
# Sanity checks: expected rows and any missing combinations.
expected_rows = len(RHO_VALUES) * N_REPEATS * len(METHODS)
actual_rows = len(pd.read_csv(DETAIL_PATH)) if DETAIL_PATH.exists() else 0
print('Expected rows:', expected_rows)
print('Actual rows:', actual_rows)

if actual_rows:
    detail = pd.read_csv(DETAIL_PATH)
    missing = []
    for rho in RHO_VALUES:
        for repeat_index in range(N_REPEATS):
            for method in METHODS:
                mask = (
                    (detail['rho'] == rho)
                    & (detail['repeat'] == repeat_index)
                    & (detail['method'] == method['name'])
                )
                if not mask.any():
                    missing.append((rho, repeat_index, method['name']))
    print('Missing combinations:', len(missing))
    if missing[:10]:
        print('First missing:', missing[:10])